# 构建会话浏览器

当你把智能体作为产品交付（一个桌面应用、一个 IDE 扩展、一个内部聊天机器人），用户最先要的就是侧边栏。他们想看到上周二的对话，跳回去继续，也许在不丢失原始内容的前提下岔出一条新方向。智能体循环只是产品的一半；另一半是会话管理。

Claude Agent SDK 会把每一次对话写入磁盘上的 JSONL 会话记录。它同时提供一组读取这些会话记录并加以组织的函数，这样你不必自己写文件解析器，也不必手工遍历 `~/.claude/projects/`，就能做出那个侧边栏。

**学完本 cookbook，你将能够：**

- 为某个项目列出并渲染历史会话，支持分页以及分支、标题、最后修改时间等元数据
- 把已保存会话的消息读回你的界面，而不启动智能体
- 重命名、打标签和筛选会话，让用户能够整理自己的历史
- 在任意位置分叉一个会话，并把分叉结果作为一次活的 `query()` 调用恢复运行

这就是 Claude Code Desktop 和 VS Code 扩展中会话侧边栏背后的模式。同样的原语适用于你想在 Agent SDK 之上搭建的任何界面。

## 前置条件

在跟随本指南之前，请确认你具备：

**所需知识**

- Python 基础，包括 `async`/`await`
- 对 Agent SDK 的 `query()` 函数有基本了解（入门可参见[笔记本 00](00_The_one_liner_research_agent.ipynb)）

**所需工具**

- Python 3.11 或更高版本
- Claude Code CLI（`npm install -g @anthropic-ai/claude-code`）
- 一个 Anthropic API 密钥（[在此获取](https://console.anthropic.com)）

## 环境准备

安装所需依赖。会话管理函数从 `claude-agent-sdk` v0.1.51 起可用。

In [ ]:
%%capture
%pip install -U "claude-agent-sdk>=0.1.51" python-dotenv pandas

从 `.env` 读取 API 密钥并配置模型。这里用 Haiku，因为演示会话很短，我们希望它们便宜且快；在真实产品里你会选用适合你智能体的模型。

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv

load_dotenv()

MODEL = "claude-haiku-4-5"

# All demo sessions live under this project directory. Using a dedicated
# cwd keeps the demo isolated from your real Claude Code sessions.
# Note: this path resolves relative to the kernel's working directory
# (claude_agent_sdk/ when launched per the README).
DEMO_DIR = str(Path("session_browser_demo").resolve())
os.makedirs(DEMO_DIR, exist_ok=True)
print(f"Demo project dir: {DEMO_DIR}")

# 第 1 部分：先创建几个待管理的会话

会话管理函数读取的是 `query()`（或 Claude Code CLI）已经写入的会话记录。为了有内容可浏览，我们先跑三段简短对话，并记下它们的会话 ID。

`cwd` 选项告诉 SDK 这段对话属于哪个项目目录。会话记录存放在 `~/.claude/projects/<encoded-cwd>/<session-id>.jsonl` 下，所以每次用同一个 `cwd` 的调用都会落到同一个桶里。那个桶就是后续 `list_sessions(directory=...)` 读取的对象。

我们关闭工具并把每次运行限制在一轮，以保持词元消耗最小。

In [ ]:
from claude_agent_sdk import ClaudeAgentOptions, ResultMessage, query


async def run_one_turn(prompt: str) -> str:
    """Run a single-turn conversation and return its session_id."""
    opts = ClaudeAgentOptions(
        model=MODEL,
        cwd=DEMO_DIR,
        max_turns=1,
        allowed_tools=[],  # text-only, no tool loop
    )
    session_id = None
    async for msg in query(prompt=prompt, options=opts):
        if isinstance(msg, ResultMessage):
            session_id = msg.session_id
            preview = (msg.result or "")[:80]
            print(f"[{session_id[:8]}] {preview}...")
    if session_id is None:
        raise RuntimeError("No ResultMessage received; check API key and SDK version.")
    return session_id

三个提示词，三个会话。在真实产品里，这些会是用户向智能体提出的内容。

In [ ]:
prompts = [
    "Give me three name ideas for a CLI tool that manages git worktrees.",
    "Explain the difference between a mutex and a semaphore in one paragraph.",
    "Write a haiku about merge conflicts.",
]

demo_session_ids = []
for p in prompts:
    sid = await run_one_turn(p)
    demo_session_ids.append(sid)

print(f"\nCreated {len(demo_session_ids)} sessions.")

# 第 2 部分：列出并查看会话

## 构建会话列表

`list_sessions()` 扫描项目的会话记录目录，为每个会话返回元数据，按最新在前排序。它读取文件统计信息加上每个会话记录的首尾，所以即使目录里有几百个文件也仍然很快。不会派生子进程，也不会发起 API 调用。

每个 `SDKSessionInfo` 都带有你在选择器里渲染一行所需的内容：用于展示的摘要、最后修改时间戳、git 分支、工作目录，以及你设置的自定义标题或标签。

In [ ]:
from datetime import datetime

import pandas as pd

from claude_agent_sdk import SDKSessionInfo, list_sessions

sessions = list_sessions(directory=DEMO_DIR)

# Render as a table. In a real UI this would be your sidebar component.
rows = []
for s in sessions:
    rows.append(
        {
            "id": s.session_id[:8],
            "summary": (s.summary[:50] + "...") if len(s.summary) > 50 else s.summary,
            "modified": datetime.fromtimestamp(s.last_modified / 1000).strftime("%H:%M:%S"),
            "branch": s.git_branch or "-",
            "tag": s.tag or "-",
        }
    )

pd.DataFrame(rows)

对于很长的历史，传入 `limit` 和 `offset` 来分页取结果。会话选择器通常先加载第一页，然后随用户滚动再取更多：

```python
page_2 = list_sessions(directory=DEMO_DIR, limit=20, offset=20)
```

如果你的应用已经存了某个会话 ID（比如存在自己的数据库里，与用户记录放在一起），只需要那一行数据，`get_session_info()` 比列出全部更省。

In [ ]:
from claude_agent_sdk import get_session_info

info = get_session_info(demo_session_ids[0], directory=DEMO_DIR)

print(f"Session:      {info.session_id}")
print(f"Summary:      {info.summary}")
print(f"First prompt: {info.first_prompt}")
print(f"Created:      {datetime.fromtimestamp(info.created_at / 1000)}")
print(f"Size:         {info.file_size:,} bytes")

## 读取会话的消息

用户在侧边栏点中某个会话后，你把它的对话加载到主视图。`get_session_messages()` 从会话记录重建消息链，按顺序返回用户和助手轮次。和列表函数一样，这是纯文件读取：智能体不必处于运行状态。

每个 `SessionMessage` 有一个 `type`（`"user"` 或 `"assistant"`）、一个 `uuid`，以及一个与 Anthropic Messages API 同形状的 `message` 字典（`role`、`content`）。

In [ ]:
from claude_agent_sdk import get_session_messages

messages = get_session_messages(demo_session_ids[0], directory=DEMO_DIR)

for m in messages:
    role = m.type
    # content is a list of blocks; pull out the text ones
    text_parts = [
        b.get("text", "")
        for b in m.message.get("content", [])
        if isinstance(b, dict) and b.get("type") == "text"
    ]
    text = " ".join(text_parts).strip()
    print(f"[{role:>9}] {text[:100]}")

对于长会话，`limit` 和 `offset` 让你一次加载一个窗口。聊天视图可能在打开时加载最后 50 条消息，然后随用户向上滚动取更早的页。偏移按时间顺序（最早在前）应用，所以第 0 页就是对话的开头。

# 第 3 部分：用标题和标签组织

## 重命名会话

自动生成的摘要适合快速浏览，但用户常常想给会话一个真正的名字。`rename_session()` 向会话记录追加一条标题条目；`list_sessions()` 在下次读取时把它作为 `custom_title` 取出。

追加操作开销低且幂等：重命名调用两次只是较新的标题生效。不会重写文件。

In [ ]:
from claude_agent_sdk import rename_session

rename_session(demo_session_ids[0], "Worktree CLI naming brainstorm", directory=DEMO_DIR)
rename_session(demo_session_ids[2], "Haiku corner", directory=DEMO_DIR)

# Verify the titles stuck
for s in list_sessions(directory=DEMO_DIR):
    label = s.custom_title or "(auto)"
    print(f"{s.session_id[:8]}  custom_title={label!r}  summary={s.summary[:40]!r}")

## 打标签与筛选

标签是附加在会话上的单个字符串。按你产品需要的分类方式使用即可：`"archived"`、`"needs-review"`、`"favorite"`。传入 `None` 可清除标签。

常见模式是软删除：不删除会话记录文件，而是给它打上 `"__hidden"` 标签，并在列表视图中过滤掉。数据仍可恢复。

In [ ]:
from claude_agent_sdk import tag_session

# Mark two sessions as favorites, hide the other
tag_session(demo_session_ids[0], "favorite", directory=DEMO_DIR)
tag_session(demo_session_ids[2], "favorite", directory=DEMO_DIR)
tag_session(demo_session_ids[1], "__hidden", directory=DEMO_DIR)


def visible_sessions(directory: str, tag_filter: str | None = None) -> list[SDKSessionInfo]:
    """List sessions, hiding soft-deletes and optionally filtering by tag."""
    results = []
    for s in list_sessions(directory=directory):
        if s.tag == "__hidden":
            continue
        if tag_filter is not None and s.tag != tag_filter:
            continue
        results.append(s)
    return results


favorites = visible_sessions(DEMO_DIR, tag_filter="favorite")
print(f"Visible favorites: {len(favorites)}")
for s in favorites:
    print(f"  {s.session_id[:8]}  [{s.tag}]  {s.custom_title or s.summary}")

标签是单值，不是列表。如果你需要多个维度（比如一个状态加一个类别），把它们编码进一个字符串如 `"review:urgent"` 并在读取时解析，或者把更丰富的状态存在你自己的数据库里，以 `session_id` 为键。

# 第 4 部分：分叉与恢复

## 从已有对话分叉

分叉会把会话的会话记录复制成一个新文件，消息 ID 重新映射。原始会话保持不变。这就是“换个思路试试”类功能背后的原语：用户保留原来的线程，同时得到一个新的用于试验。

`fork_session()` 写入新文件并返回它的 ID。它不会运行智能体，所以分叉结果留在磁盘上，直到你用 `query()` 恢复它。

In [ ]:
from claude_agent_sdk import fork_session

source = demo_session_ids[0]

fork = fork_session(
    source,
    directory=DEMO_DIR,
    title="Worktree CLI names (round 2)",
)

print(f"Source: {source}")
print(f"Fork:   {fork.session_id}")

# The fork starts with the same message history as the source
source_msgs = get_session_messages(source, directory=DEMO_DIR)
fork_msgs = get_session_messages(fork.session_id, directory=DEMO_DIR)
print(f"Source has {len(source_msgs)} messages, fork has {len(fork_msgs)}")

要从某个特定点而不是完整历史分叉，传入 `up_to_message_id`。分叉结果包含源会话记录直到并包含该消息的部分。你可以从 `get_session_messages()[i].uuid` 取得消息 UUID。

## 把分叉恢复为一次活的 query

分叉只是一个会话记录文件。要把它变回运行中的对话，把它的 ID 交给 `ClaudeAgentOptions.resume`。智能体加载分叉的历史并从那里继续。

In [ ]:
resume_opts = ClaudeAgentOptions(
    model=MODEL,
    cwd=DEMO_DIR,
    max_turns=1,
    allowed_tools=[],
    resume=fork.session_id,
)

async for msg in query(
    prompt="Those were okay. Give me three more names, but punnier.",
    options=resume_opts,
):
    if isinstance(msg, ResultMessage):
        print(f"[fork {fork.session_id[:8]} resumed]")
        print(msg.result)

原始会话仍然原封不动地在那里。再次列出，你会看到源和分叉作为两行各自独立的历史。

In [ ]:
for s in list_sessions(directory=DEMO_DIR):
    marker = "(fork)" if s.session_id == fork.session_id else "      "
    print(f"{marker} {s.session_id[:8]}  {s.custom_title or s.summary[:50]}")

# 清理

`delete_session()` 删除一个会话记录文件。这是硬删除，正因如此第 3 部分的软删除标签模式通常是面向用户界面更安全的默认做法。

这里我们用它清理演示创建的所有内容。

In [ ]:
from claude_agent_sdk import delete_session

# Clean up every session in the demo dir, including the fork
for s in list_sessions(directory=DEMO_DIR):
    delete_session(s.session_id, directory=DEMO_DIR)
    print(f"Deleted {s.session_id[:8]}")

remaining = list_sessions(directory=DEMO_DIR)
print(f"\n{len(remaining)} session(s) remaining.")

# 小结

我们围绕 Agent SDK 的本地会话记录存储构建了会话浏览器的核心：

- **列出**：`list_sessions()` 给出渲染侧边栏所需的一切，并且能扩展，因为它读取文件统计信息和首尾片段，而不是解析整份会话记录。
- **读取**：`get_session_messages()` 把对话加载回来用于展示，而不启动智能体。
- **组织**：`rename_session()` 和 `tag_session()` 追加元数据条目，所以开销低，且最近一次调用生效。
- **分叉**：`fork_session()` 加上 `options.resume` 让用户分出一条对话并继续，而不触碰原始会话。

这些都是对 `~/.claude/projects/` 的纯文件操作。无论智能体子进程是否在运行它们都有效，并且它们看到的正是 Claude Code CLI 写入的会话记录。

## 接下来去哪里

- **接到界面上。** 这些函数与界面无关；把它们放到一个 FastAPI 路由或 Electron IPC 处理函数后面，你就有了会话侧边栏的后端。
- **跨主机会话。** 会话记录存在本地磁盘上。要在多台机器之间共享会话，就同步这些文件，或者把会话 ID 和消息索引到你自己的存储中。模式参见[在磁盘上管理会话](https://docs.claude.com/en/agent-sdk/local-session-management)。
- **TypeScript。** 同样的 API 存在于 `@anthropic-ai/claude-agent-sdk`，用驼峰命名（`listSessions`、`forkSession` 等）。参见 [TypeScript SDK 参考](https://docs.claude.com/en/agent-sdk/typescript)。
- **更大的图景。** [笔记本 00](00_The_one_liner_research_agent.ipynb) 到 [03](03_The_site_reliability_agent.ipynb) 讲解如何构建智能体本身。本笔记本讲解如何管理它留下的东西。